# 🏠 House Card Asset Generator

Generates a consistent set of isometric house images for the
[House Card](https://github.com/muhlman/house_card) (a fork of
[silasmariusz/fork_u-house_card](https://github.com/silasmariusz/fork_u-house_card)).

**What it makes**
- Base images for all 4 seasons, day and night (8).
- Weather variants: overcast, rain, snow, fog, lightning, hail - day and night, every season (48, or 44 with the unlikely combinations skipped).
- Two Christmas images (14 Dec - 14 Jan).
- Optional: up to two **vehicle overlays** (transparent PNGs placed on the driveway while a person is home), with day/night and weather variants. Each is added to your finished house image as a local edit and cut out by difference, so it lands in the right place at the right scale.
- Optional: **event decorations** (4th of July, birthday, Halloween, Thanksgiving, New Year, Easter, ...) as transparent overlays shown by date window, so the real weather stays underneath - or as full house renders if you prefer.
- Optional: the upstream "gaming mode" images (the card does not use them).

Every output is resized to the master's pixel size so the card and the overlays line up exactly. Vehicles are also saved as cropped **sprites** with the x / y / width values to position them from the card config.

### Instructions
1. **API key**: create one at <https://aistudio.google.com/apikey> and paste it in step 1. The free tier is enough.
2. **Reference photos**: step 2 asks for 1-5 photos of your house (street view, side view, satellite / roof).
3. **Master**: step 4 renders the summer-day master. Re-run it until you like it - every other image copies its geometry.
4. **Run the rest**: steps 5-7b generate, step 8 shows thumbnails, step 9 downloads a zip. Already-generated files are skipped, so you can re-run after a rate limit or delete a file to redo it.
5. Copy the PNGs to the folder your card's `image_path` points at (default `config/www/community/house_card/images/`).


In [ ]:
#@title 1. Setup & Configuration
#@markdown API key: either paste it below, or store it once in Colab's **Secrets** (key icon in the left sidebar) as
#@markdown `GOOGLE_API_KEY` with notebook access enabled - then leave the field empty and it is picked up automatically.
#@markdown Create a key at https://aistudio.google.com/apikey.

import io, os, re, time, zipfile
from pathlib import Path
from google.colab import files, userdata

!pip install -q -U google-genai
# opencv (preinstalled on Colab) is used to align edited renders to their base before extracting overlays
from google import genai
from google.genai import types
from PIL import Image
import logging, warnings
logging.getLogger("google_genai.models").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=DeprecationWarning)

GOOGLE_API_KEY = "" #@param {type:"string"}
if not GOOGLE_API_KEY:
    try:
        GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY") or ""
        if GOOGLE_API_KEY: print("🔑 Using GOOGLE_API_KEY from Colab Secrets.")
    except Exception:  # noqa: BLE001 - no secret, or notebook access not granted
        pass
HOUSE_DESCRIPTION = "White two-story colonial with black shutters, red brick base, attached two-car garage on the right, paver driveway" #@param {type:"string"}

#@markdown **Model.** `gemini-nano-banana-2.1` is the current fast model (1K/2K/4K); `gemini-3-pro-image` is the higher-fidelity Nano Banana Pro; `gemini-2.5-flash-image` is the older free-tier model.
MODEL_TYPE = "gemini-nano-banana-2.1" #@param ["gemini-nano-banana-2.1", "gemini-3-pro-image", "gemini-3.1-flash-image", "gemini-2.5-flash-image"]
#@markdown **Frame.** Set the card's `aspect_ratio` to the same value (the card defaults to 4:3). 1K is plenty for a dashboard.
ASPECT_RATIO = "3:2" #@param ["3:2", "4:3", "16:9", "1:1"]
IMAGE_SIZE = "1K" #@param ["1K", "2K", "4K"]

#@markdown **What to generate.**
GENERATE_RARE_WEATHER = False #@param {type:"boolean"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Also make summer snow and winter hail. Off by default: the card falls back to the base image when a file is missing.
GENERATE_XMAS = True #@param {type:"boolean"}
GENERATE_EVENTS = False #@param {type:"boolean"}
EVENTS_TO_GENERATE = "july4, birthday, halloween, thanksgiving, new_year" #@param {type:"string"}
EVENT_MODE = "overlay" #@param ["overlay", "full"]
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Available: july4, birthday, halloween, thanksgiving, new_year, easter, valentines, st_patricks, graduation, game_day.
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;**overlay** (recommended): just the decorations as a transparent PNG, day and night, shown by the card's
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;`overlays:` with a date window - the real weather image stays underneath, so two files cover every weather.
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;**full**: complete house renders for the card's `events:` (one scene, no weather variation). Birthday is made per season.
GENERATE_GAMING = False #@param {type:"boolean"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Upstream "gaming mode" images; the card does not display them.

#@markdown **Vehicle overlays.** Leave a description empty to skip that vehicle. The house images are then
#@markdown generated with an **empty driveway** so the overlays always land on bare pavement.
VEHICLE_1_DESCRIPTION = "black Ford F-150 pickup truck, gloss black" #@param {type:"string"}
VEHICLE_1_PARKING_SPOT = "on the paver driveway in front of the LEFT garage door, nose pointing away from the garage" #@param {type:"string"}
VEHICLE_2_DESCRIPTION = "white Lexus NX 350 SUV, pearl white" #@param {type:"string"}
VEHICLE_2_PARKING_SPOT = "on the paver driveway in front of the RIGHT garage door, nose pointing away from the garage" #@param {type:"string"}
VEHICLE_WEATHER_VARIANTS = True #@param {type:"boolean"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Off = only day and night per vehicle (2 files). On = also overcast, rainy, snowy, fog, lightning, hail (12 files).
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Give each vehicle its own spot so both can show at once without overlapping. Describe the spot relative to
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;fixed features of the master image (garage doors, front walk, mailbox).
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Optional regions (`left,top,right,bottom` in % of the frame) limit where each vehicle may be extracted from.
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Leave empty to keep the largest changed area; step 8 prints the region each vehicle was found in.
VEHICLE_1_REGION = "" #@param {type:"string"}
VEHICLE_2_REGION = "" #@param {type:"string"}
EXTRACT_METHOD = "model" #@param ["model", "diff"]
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;How overlays are cut out. **model**: a second edit asks the model to paint everything except the added object solid
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;magenta, which is then keyed to transparency (one extra API call per overlay, cleanest result). **diff**: compare the edited
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;render with its base after aligning them (free, but misses parts of objects that match the background colour).

def _region(txt):
    nums = [float(x) for x in re.findall(r"[\d.]+", txt or "")]
    return tuple(nums) if len(nums) == 4 else None
VEHICLES = [(d.strip(), s.strip(), _region(r)) for d, s, r in
            ((VEHICLE_1_DESCRIPTION, VEHICLE_1_PARKING_SPOT, VEHICLE_1_REGION),
             (VEHICLE_2_DESCRIPTION, VEHICLE_2_PARKING_SPOT, VEHICLE_2_REGION)) if d.strip()]
EMPTY_DRIVEWAY = len(VEHICLES) > 0

if not GOOGLE_API_KEY:
    raise SystemExit("❌ No API key. Paste one in the form above, or add a Colab Secret named GOOGLE_API_KEY and allow notebook access.")
client = genai.Client(api_key=GOOGLE_API_KEY)
EVENTS_SELECTED = [e.strip().lower() for e in EVENTS_TO_GENERATE.split(",") if e.strip()] if GENERATE_EVENTS else []
print(f"✅ Model {MODEL_TYPE}, {ASPECT_RATIO} at {IMAGE_SIZE}; vehicles: {len(VEHICLES)}; empty driveway: {EMPTY_DRIVEWAY}; events: {EVENTS_SELECTED or 'none'}")

BASE_DIR = Path("/content/house_assets")
REF_DIR, OUT_DIR, MASTER_DIR, VEHICLE_DIR, SPRITE_DIR = (BASE_DIR / d for d in ("reference", "output", "master", "output/overlays", "output/sprites"))
for d in (REF_DIR, OUT_DIR, MASTER_DIR, VEHICLE_DIR, SPRITE_DIR):
    d.mkdir(parents=True, exist_ok=True)
MASTER_PATH = MASTER_DIR / "_master_reference.png"


In [ ]:
#@title 2. Upload Reference Images
#@markdown Upload 1-5 photos of your house. Ideally: 1 front view, 1 side view, 1 isometric/satellite view if available.

uploaded = files.upload()

for filename in uploaded.keys():
    # Move to reference dir
    os.rename(filename, REF_DIR / filename)
    print(f"Saved reference: {filename}")

print(f"✅ Total references: {len(list(REF_DIR.glob('*')))}")

In [ ]:
#@title 3. Prompts & generation helpers

SEASONS = ["winter", "spring", "summer", "autumn"]
TIMES = ["day", "night"]
# Suffixes exactly as the card requests them ({season}_{weather}_{time}.png)
WEATHERS = ["overcast", "rainy", "snowy", "fog", "lightning", "hail"]
UNLIKELY = {("summer", "snowy"), ("winter", "hail")}

WEATHER_RULES = {
    "sunny":     "Clear sky: plot highly illuminated by golden sun rays (day) or a crisp clear starry night.",
    "overcast":  "Overcast: flat grey cloud cover, soft diffuse light, no sun shafts, muted colours. No rain.",
    "rainy":     "Heavy rain: a downpour over the plot, wet reflective surfaces, puddles with light reflections.",
    "snowy":     "Snowing: intense snowfall over the plot, fresh snow accumulating on roof, ground and surfaces.",
    "hail":      "Hailstorm: large chunks of ice bouncing and lying on the ground and roof instead of snow, random puddles under the ice.",
    "lightning": "Thunderstorm: dark storm sky, the house strongly overexposed by lightning flashes, hard low-angle shadows. Do not draw lightning bolts.",
    "fog":       "Fog: a delicate cloud layer and slight haze in front of the house and across the plot, softened contrast.",
}
SEASON_GROUND = {
    "winter": "Winter: ground and roof covered in snow, bare trees. No grass.",
    "spring": "Spring: fresh green grass with a few spring flowers, trees in light leaf.",
    "summer": "Summer: manicured vibrant green lawn, full foliage.",
    "autumn": "Autumn: a moderate scatter of yellow-orange-brown leaves on the grass, trees in autumn colour.",
}
GAMING_MODES = {
    "synthwave": "Synthwave style: neon purples/pinks, grid lines, retro-future vibe. Flying DeLorean parked in front.",
    "cyberpunk": "Cyberpunk style: high-tech low-life, neon signs, rain, dark gritty atmosphere. Flying DeLorean parked in front.",
    "matrix":    "Matrix style: green code rain, digital artifacts. Half the asset in green wireframe vector style.",
    "mario":     "Mario Bros World style: green pipe, gold coins/stars on roof, beanstalk to sky. Mario & Luigi grilling in front of the house.",
    "xbox_kid":  "Small astronaut kid (big head) playing Xbox on a red semi-transparent sofa in front of the house. Projector screen.",
}
# Event images: (season the scene is set in, or None = one per season; prompt extras)
EVENTS = {
    "july4":        ("summer", "4th of July: American flags and red-white-blue bunting on the porch and railings, a flag on a pole, "
                               "fireworks bursting in the sky above the house, a few lawn chairs and a small grill on the patio, sparkler light trails."),
    "birthday":     (None,     "Birthday party: colourful balloons tied to the porch, mailbox and railings, a 'HAPPY BIRTHDAY' banner across the front, "
                               "streamers, a table with a birthday cake and presents on the patio, confetti on the lawn."),
    "halloween":    ("autumn", "Halloween: carved glowing jack-o'-lanterns on the steps, cobwebs on the porch, a skeleton by the door, "
                               "orange and purple string lights, a few gravestone props on the lawn, bats in the sky."),
    "thanksgiving": ("autumn", "Thanksgiving: harvest decorations - pumpkins, gourds and corn stalks by the door, a wreath of autumn leaves, "
                               "warm light in every window, a long table set for dinner visible through the windows."),
    "new_year":     ("winter", "New Year's Eve: fireworks and confetti over the house, a glowing '2026' style numerals sign on the lawn, champagne-gold "
                               "string lights, party hats and streamers on the porch, snow on the ground."),
    "easter":       ("spring", "Easter: pastel-coloured eggs hidden across the lawn and flower beds, a large bunny figure by the porch, "
                               "pastel bunting, spring flowers in bloom, a basket of eggs on the steps."),
    "valentines":   ("winter", "Valentine's Day: red and pink heart decorations and a heart wreath on the door, pink string lights along the eaves, "
                               "rose petals on the path, warm romantic lighting."),
    "st_patricks":  ("spring", "St. Patrick's Day: green bunting and shamrock decorations, a rainbow ending in a pot of gold on the lawn, "
                               "green string lights, a leprechaun hat on the mailbox."),
    "graduation":   ("spring", "Graduation party: a 'CONGRATS GRAD' banner across the porch, balloons in school colours, a mortarboard and "
                               "diploma on the lawn sign, a table with cake and a photo board on the patio."),
    "game_day":     ("autumn", "Game day: team pennants and a 'GAME DAY' banner, a tailgate setup on the driveway edge with a grill, "
                               "cooler and folding chairs, a projector screen on the garage showing a football field, foam fingers."),
}

XMAS_EXTRAS = {
    "day":   "Christmas: Santa sliding down the roof, penguins, an igloo in the garden, festive decorations and wreaths.",
    "night": "Christmas: Santa sliding down the roof, penguins, an igloo in the garden, strings of glowing Christmas lights on the eaves.",
}

DRIVEWAY_RULE = ("- DRIVEWAY: completely EMPTY. No vehicles, bins, planters or objects on the driveway or in front of the garage. "
                 "Bare pavement only." if EMPTY_DRIVEWAY else "")

BASE_RULES = f"""
RULES (CAMERA - MANDATORY):
- VIEW: STRICT ISOMETRIC ONLY (SimCity / The Sims style). Equal angles, parallel lines stay parallel.
- The house and plot are shown in this fixed isometric projection. NEVER render from a different angle.
- MARGINS: 15% left, 15% right, 5% top, 15% bottom. The asset is centred within these margins with the whole glassmorphism base visible. Weather effects may extend beyond the margins.
- ASPECT RATIO: {ASPECT_RATIO}.
{DRIVEWAY_RULE}

RULES (VISUAL STYLE):
- Ultra-high-fidelity 3D asset, "SimCity 2030" / "The Sims 5" aesthetic. Dramatic, cinematic, high-contrast lighting.
- Volumetric light only on the asset, never on the background. Soft bloom, ray-traced reflections on glass, wet surfaces and paint.
- A thick multi-layered glassmorphism base plate at the very bottom, with soil/grass/snow on top.
- Background: solid #121212 everywhere outside the plot, for a clean cut-out. Nothing drawn on the background.
- Quality: maximum sharpness, zero blur, no text, no watermark.

RULES (FIDELITY):
- House: {HOUSE_DESCRIPTION}
- CONSISTENCY: geometry, camera angle, position and scale 100% identical to the master reference. Use references only for geometry and architectural detail; re-draw the scene, do not filter the reference.
"""

def master_prompt():
    return f"""Task: create a "SimCity 2030" style 3D isometric asset of this house from the attached photos.
{BASE_RULES}
This image: Summer, Day, {WEATHER_RULES['sunny']} {SEASON_GROUND['summer']}
"""

def event_prompt(season, time_of_day, extras):
    """Full-scene event render (EVENT_MODE = full)."""
    return variant_prompt(season, time_of_day, "sunny", "Event decorations (keep the house itself unchanged): " + extras)

def event_overlay_prompt(time_of_day, extras):
    """Local edit of a finished house image: add decorations only (EVENT_MODE = overlay)."""
    light = ("Night: decorations and string lights glow; match the scene's blue-grey night ambient and lamp light."
             if time_of_day == "night" else "Match the scene's daylight and shadow direction exactly.")
    return f"""Edit the attached image. Add these decorations to the scene: {extras}
Place them on and around the house, porch, lawn, path and (for fireworks or sky effects) in the dark sky area above the house.
Change NOTHING else: the house, roof, trees, lawn, driveway, lighting and background must stay pixel-identical to the attached image. Do not move the camera, do not re-render the scene, do not add vehicles or people unless listed.
{light}
Same style as the scene (ultra-high-fidelity 3D asset). No text other than banners that are part of the decorations, no watermark.
"""

def variant_prompt(season, time_of_day, weather="sunny", extras=""):
    night = "Night: deep navy / blue-grey grading on the whole plot, warm lights on inside the house, driveway and path lamps lit." if time_of_day == "night" else "Day."
    return f"""Reference: the attached master image. Same isometric angle, position and scale - redraw it for the conditions below.
{BASE_RULES}
This image:
- Season: {season.capitalize()}. {SEASON_GROUND[season]}
- Time: {night}
- Weather: {WEATHER_RULES[weather]}
{extras}
"""

def vehicle_prompt(desc, spot, time_of_day, weather):
    """Local edit of a finished house image: the model keeps the scene and adds the vehicle."""
    wx = "" if weather == "sunny" else (f"Weather effect on the vehicle only: {WEATHER_RULES[weather]} "
                                         "Show it on the vehicle (wet paint, snow or ice on roof and hood, haze, flash highlights).")
    light = ("Night: the vehicle is lit by the driveway lamps and house lights with the same blue-grey ambient as the scene."
             if time_of_day == "night" else "Match the scene's daylight and shadow direction exactly.")
    return f"""Edit the attached image. Add ONE {desc}, parked {spot}, drawn at the exact scale and isometric angle of the scene, with a soft contact shadow under it.
Change NOTHING else: the house, roof, trees, lawn, driveway, lighting, sky and background must stay pixel-identical to the attached image. Do not move the camera, do not re-render the scene, do not add other objects.
{light}
{wx}
Same style as the scene (ultra-high-fidelity 3D asset, gloss paint with ray-traced reflections). No text, no watermark.
"""

def _config():
    """image_config is the documented way to pin aspect ratio / size; fall back if the SDK or model rejects it."""
    no_afc = dict(automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
    attempts = [
        dict(response_modalities=["IMAGE"], image_config=types.ImageConfig(aspect_ratio=ASPECT_RATIO, image_size=IMAGE_SIZE)),
        dict(response_modalities=["IMAGE"], image_config=types.ImageConfig(aspect_ratio=ASPECT_RATIO)),
        dict(response_modalities=["IMAGE"]),
    ]
    for kw in attempts:
        try:
            yield types.GenerateContentConfig(**kw, **no_afc)
        except (TypeError, ValueError):
            continue

def _extract_image(response):
    for part in getattr(response, "parts", None) or (response.candidates[0].content.parts if response.candidates else []):
        if getattr(part, "inline_data", None) and part.inline_data.data:
            return Image.open(io.BytesIO(part.inline_data.data))
    return None

def generate_image(prompt, ref_paths, out_path, match_master=True, retries=3):
    """Generate one image, resize it to the master's size (so every file lines up), save as PNG."""
    contents = [prompt] + [Image.open(p) for p in ref_paths if Path(p).exists()]
    for attempt in range(retries):
        for cfg in _config():
            try:
                response = client.models.generate_content(model=MODEL_TYPE, contents=contents, config=cfg)
            except Exception as e:  # noqa: BLE001
                msg = str(e)
                if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                    print("   rate limit - waiting 60 s"); time.sleep(60); break
                if "400" in msg and ("image_config" in msg or "image_size" in msg or "aspect" in msg):
                    continue  # try the next, simpler config
                print(f"   error: {msg[:200]}"); break
            img = _extract_image(response)
            if img is None:
                print("   no image in response"); break
            if match_master and MASTER_PATH.exists() and out_path != MASTER_PATH:
                size = Image.open(MASTER_PATH).size
                if img.size != size:
                    img = img.resize(size, Image.LANCZOS)
            img.convert("RGB").save(out_path, "PNG")
            return True
    return False

def shape_prior(overlay_path, grow_pct=3.0):
    """A box around an existing overlay (alpha > 0), grown by grow_pct of the frame. Used to confine a
    weather variant to where the clear-weather vehicle is, with room for snow on the roof."""
    import numpy as np
    a = np.asarray(Image.open(overlay_path).convert("RGBA"))[..., 3] > 0
    if not a.any(): return None
    H, W = a.shape
    ys, xs = np.where(a); gy, gx = int(H * grow_pct / 100), int(W * grow_pct / 100)
    box = np.zeros_like(a); box[max(0, ys.min() - gy):ys.max() + gy + 1, max(0, xs.min() - gx):xs.max() + gx + 1] = True
    return box

def align_to_base(edit, base):
    """The model re-renders the frame with slightly different framing (a pixel or two of shift / scale), which
    makes every edge in the picture look changed. Estimate an affine warp (ECC, on a downscaled grey copy)
    and warp the edited image onto the base. Returns (aligned PIL image, description) or (edit, None)."""
    import numpy as np
    try:
        import cv2
    except ImportError:
        return edit, None
    g_base = cv2.cvtColor(np.asarray(base), cv2.COLOR_RGB2GRAY)
    g_edit = cv2.cvtColor(np.asarray(edit), cv2.COLOR_RGB2GRAY)
    scale = 640 / max(g_base.shape)
    s_base = cv2.resize(g_base, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    s_edit = cv2.resize(g_edit, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    warp = np.eye(2, 3, dtype=np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 300, 1e-6)
    try:
        cc, warp = cv2.findTransformECC(s_base, s_edit, warp, cv2.MOTION_AFFINE, criteria, None, 5)
    except cv2.error:
        return edit, "alignment failed - using the image as is"
    warp[:, 2] /= scale
    H, W = g_base.shape
    aligned = cv2.warpAffine(np.asarray(edit), warp, (W, H), flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP, borderMode=cv2.BORDER_REPLICATE)
    sx, sy = float(np.hypot(warp[0, 0], warp[1, 0])), float(np.hypot(warp[0, 1], warp[1, 1]))
    desc = f"shift ({warp[0, 2]:+.1f}, {warp[1, 2]:+.1f}) px, scale {sx:.3f}x{sy:.3f}"
    return Image.fromarray(aligned), desc

def extract_changes(edited_path, base_path, dst_path, region=None, keep="largest", threshold=28, min_area_pct=0.15,
                    within=None, blur=0, clean=7, align=True):
    """Alpha-mask the pixels that changed between the base scene and the edited scene.
    keep="largest": one object (a vehicle) - the largest changed blob, or the blobs inside `region` (% of frame).
    keep="all": every changed blob above min_area_pct (decorations scattered around the house).
    within: optional boolean mask (see shape_prior) outside which changes are ignored.
    blur: Gaussian radius applied to both images before comparing - re-render jitter in foliage, brick and
          grass averages out while solid added objects remain. clean: speck-removal kernel (odd px).
    Writes a full-frame RGBA PNG. Returns (coverage %, bbox %)."""
    import numpy as np
    from PIL import ImageFilter
    base = Image.open(base_path).convert("RGB")
    edit = Image.open(edited_path).convert("RGB")
    if edit.size != base.size:
        edit = edit.resize(base.size, Image.LANCZOS)
    align_desc = None
    if align:
        edit, align_desc = align_to_base(edit, base)
        if align_desc: print(f"   aligned: {align_desc}")
    cmp_base, cmp_edit = (base.filter(ImageFilter.GaussianBlur(blur)), edit.filter(ImageFilter.GaussianBlur(blur))) if blur else (base, edit)
    a = np.asarray(cmp_base).astype(np.int16); b = np.asarray(cmp_edit).astype(np.int16)
    diff = np.abs(a - b).max(axis=2)
    mask = diff > threshold
    if within is not None:
        mask &= within
    # open (drop specks) then close (fill small gaps)
    m = Image.fromarray((mask * 255).astype(np.uint8)).filter(ImageFilter.MinFilter(clean)).filter(ImageFilter.MaxFilter(clean))
    m = m.filter(ImageFilter.MaxFilter(clean)).filter(ImageFilter.MinFilter(clean))
    mask = np.asarray(m) > 127
    H, W = mask.shape
    try:
        from scipy import ndimage
        labels, n = ndimage.label(mask)
        if n:
            sizes = ndimage.sum(mask, labels, range(1, n + 1))
            keep_lbl = np.zeros(n + 1, bool)
            min_px = min_area_pct / 100 * H * W
            if keep == "all":
                keep_lbl[1:] = sizes >= min_px
                if region:
                    l, t, r, btm = (region[0] * W / 100, region[1] * H / 100, region[2] * W / 100, region[3] * H / 100)
                    for i, sl in enumerate(ndimage.find_objects(labels), 1):
                        ys, xs = sl
                        if not (xs.stop > l and xs.start < r and ys.stop > t and ys.start < btm):
                            keep_lbl[i] = False
            elif region:
                l, t, r, btm = (region[0] * W / 100, region[1] * H / 100, region[2] * W / 100, region[3] * H / 100)
                for i, sl in enumerate(ndimage.find_objects(labels), 1):
                    ys, xs = sl
                    if xs.stop > l and xs.start < r and ys.stop > t and ys.start < btm and sizes[i - 1] > min_px:
                        keep_lbl[i] = True
            if not keep_lbl.any():
                keep_lbl[int(np.argmax(sizes)) + 1] = True
            mask = keep_lbl[labels]
    except ImportError:
        if region:
            l, t, r, btm = (int(region[0] * W / 100), int(region[1] * H / 100), int(region[2] * W / 100), int(region[3] * H / 100))
            box = np.zeros_like(mask); box[t:btm, l:r] = True; mask &= box
    alpha = Image.fromarray((mask * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(1.2))
    out = Image.new("RGBA", base.size)
    out.paste(edit, (0, 0)); out.putalpha(alpha)
    out.save(dst_path, "PNG")
    ys, xs = np.where(mask)
    bbox = (round(xs.min() / W * 100), round(ys.min() / H * 100), round(xs.max() / W * 100), round(ys.max() / H * 100)) if len(xs) else None
    return float(mask.mean()) * 100, bbox

MATTE = (255, 0, 255)
def matte_prompt(keep_what):
    """Second pass: the model erases the scene around the object it just added. Editing its own output keeps the
    object where it is, so nothing has to be aligned or compared."""
    return f"""Edit the attached image. Keep ONLY {keep_what}, pixel-identical in position, size, colours and detail, including its soft shadow on the ground.
Replace EVERYTHING else - the house, roof, walls, windows, garage doors, trees, bushes, lawn, the driveway pavement under and around the object, path, fence, sky and background - with one flat, solid, uniform colour: pure magenta #FF00FF. The ground is NOT part of the object: no pavement, grass or snow may remain, only the object's own soft shadow.
No gradients, no texture, no outlines, no other objects. The result is {keep_what} floating on a flat magenta field.
Do not reflect or tint the magenta onto the object's paint, glass or chrome - keep its original reflections and colours exactly.
Do not move the camera, do not resize or move anything.
"""

def key_matte(matte_path, dst_path, within=None, tolerance=70, feather=24, fill=15, edge=31):
    """Turn the magenta field into transparency. `within` (bool mask) drops anything the model forgot to erase
    far from the object. Inside the object alpha is 1; in a band along its edge, alpha follows how dominant the
    magenta is (a half-magenta shadow pixel becomes half transparent) and the magenta cast is removed, so soft
    shadows key out cleanly while pink or purple objects in the interior keep their colour.
    Writes full-frame RGBA. Returns (coverage %, bbox %)."""
    import numpy as np
    from PIL import ImageFilter
    rgb = np.asarray(Image.open(matte_path).convert("RGB")).astype(np.float32)
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    dist = np.sqrt((r - MATTE[0]) ** 2 + (g - MATTE[1]) ** 2 + (b - MATTE[2]) ** 2)
    cast = np.clip(np.minimum(r, b) - g, 0, None)          # how magenta a pixel is (0 for any grey/black/white)
    # Clearly non-magenta pixels are the object. Magenta-tinted pixels (a reflection in glass or paint, or a
    # shadow blending into the field) count only when they sit close to clear object pixels; a tinted slab
    # extending away from the body is background. The flat field (cast ~240) never qualifies.
    strong = cast < 40
    near = np.asarray(Image.fromarray((strong * 255).astype(np.uint8)).filter(ImageFilter.MaxFilter(31))) > 127
    hard = (cast < 150) & (strong | near)
    if within is not None:
        hard &= within
    # anything fully enclosed by the object (a tinted window, a dark panel) belongs to it
    try:
        from scipy import ndimage
        hard = ndimage.binary_fill_holes(hard)
    except ImportError:
        pass
    m = Image.fromarray((hard * 255).astype(np.uint8)).filter(ImageFilter.MaxFilter(fill)).filter(ImageFilter.MinFilter(fill))
    m = m.filter(ImageFilter.MinFilter(5)).filter(ImageFilter.MaxFilter(5))
    hard = np.asarray(m) > 127
    interior = np.asarray(Image.fromarray((hard * 255).astype(np.uint8)).filter(ImageFilter.MinFilter(edge))) > 127
    band = hard & ~interior
    # alpha: 1 in the interior; in the edge band fade out with magenta dominance (cast 30 -> opaque, 140 -> clear)
    edge_alpha = np.clip(1 - (cast - 30) / 110, 0, 1)
    alpha = np.where(interior, 1.0, np.where(band, edge_alpha, 0.0)).astype(np.float32)
    alpha = np.asarray(Image.fromarray((alpha * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(0.8))).astype(np.float32) / 255
    # despill: every kept pixel with a magenta cast loses it (reflections go neutral, shadows go dark)
    spill = (alpha > 0) & (cast > 20)
    r = np.where(spill, g, r); b = np.where(spill, g, b)
    out = np.dstack([r, g, b, alpha * 255]).astype(np.uint8)
    Image.fromarray(out).save(dst_path, "PNG")
    ys, xs = np.where(alpha > 0.5)
    H, W = alpha.shape
    bbox = (round(xs.min() / W * 100), round(ys.min() / H * 100), round(xs.max() / W * 100), round(ys.max() / H * 100)) if len(xs) else None
    return float((alpha > 0.5).mean()) * 100, bbox

def change_region(edited_path, base_path, grow_pct=4.0, keep="largest", region=None):
    """Where did the edit change things? A generous, grown mask from the aligned diff, used to confine the
    model matte to the right part of the frame."""
    import numpy as np, tempfile, os
    from PIL import ImageFilter
    tmp = tempfile.mktemp(suffix=".png")
    try:
        extract_changes(edited_path, base_path, tmp, region=region, keep=keep, blur=2.0, threshold=24, min_area_pct=0.02, clean=5)
        a = np.asarray(Image.open(tmp).convert("RGBA"))[..., 3] > 0
    finally:
        if os.path.exists(tmp): os.remove(tmp)
    if not a.any(): return None
    H, W = a.shape
    if keep == "largest":
        # one object: a box around everything the diff found, grown generously - the diff is patchy on
        # dark-on-dark but its extent is right, and the matte already separates object from scene
        ys, xs = np.where(a)
        gy, gx = int(H * grow_pct * 1.5 / 100), int(W * grow_pct * 1.5 / 100)
        box = np.zeros_like(a); box[max(0, ys.min() - gy):ys.max() + gy + 1, max(0, xs.min() - gx):xs.max() + gx + 1] = True
        return box
    k = max(3, int(H * grow_pct * 1.5 / 100) | 1)
    return np.asarray(Image.fromarray((a * 255).astype(np.uint8)).filter(ImageFilter.MaxFilter(k))) > 127

def extract_overlay(raw, base, dst, keep_what, matte_path, keep="largest", region=None, within=None, reuse_matte=False):
    """EXTRACT_METHOD dispatcher. Returns (coverage %, bbox %, note)."""
    if EXTRACT_METHOD == "model":
        if not (reuse_matte and matte_path.exists()):
            if not generate_image(matte_prompt(keep_what), [raw], matte_path):
                return None, None, "matte render failed"
        where = change_region(raw, base, keep=keep, region=region)
        if within is not None and where is not None:
            where &= within
        cov, bbox = key_matte(matte_path, dst, within=where)
        return cov, bbox, "model matte" + ("" if where is not None else " (no change region found - unconfined)")
    kw = dict(keep="largest", within=within, blur=1.5, threshold=32) if keep == "largest" else dict(keep="all", threshold=40, min_area_pct=0.02, blur=2.0, clean=7)
    cov, bbox = extract_changes(raw, base, dst, region=region, **kw)
    return cov, bbox, "aligned diff"

def make_sprites(prefix, pad_pct=1.5):
    """Crop every full-frame overlay named {prefix}*.png to one shared box (the union of their opaque
    areas, padded) so all variants of a vehicle share the same anchor. Writes output/sprites/<same name>.png
    and returns the card config values (x, y centre and width, in % of the frame)."""
    import numpy as np
    files = sorted(VEHICLE_DIR.glob(f"{prefix}*.png"))
    if not files: return None
    union = None
    for f in files:
        a = np.asarray(Image.open(f).convert("RGBA"))[..., 3] > 0
        ys, xs = np.where(a)
        if not len(xs): continue
        box = [xs.min(), ys.min(), xs.max(), ys.max()]
        union = box if union is None else [min(union[0], box[0]), min(union[1], box[1]), max(union[2], box[2]), max(union[3], box[3])]
    if union is None: return None
    W, H = Image.open(files[0]).size
    px, py = int(W * pad_pct / 100), int(H * pad_pct / 100)
    l, t, r, b = max(0, union[0] - px), max(0, union[1] - py), min(W, union[2] + px + 1), min(H, union[3] + py + 1)
    for f in files:
        Image.open(f).convert("RGBA").crop((l, t, r, b)).save(SPRITE_DIR / f.name, "PNG")
    return {"x": round(float(l + r) / 2 / W * 100, 1), "y": round(float(t + b) / 2 / H * 100, 1), "width": round(float(r - l) / W * 100, 1)}

print("✅ Prompts and helpers ready.")


In [ ]:
#@title 4. Generate the master reference (Summer, Day)
#@markdown Everything else copies this image's geometry, so re-run until you are happy with it.
refs = sorted(p for p in REF_DIR.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"))
if not refs:
    print("⚠️ No reference photos uploaded - the model will invent a house from the description.")
REGENERATE_MASTER = False #@param {type:"boolean"}
if MASTER_PATH.exists() and not REGENERATE_MASTER:
    print("Master already exists - tick REGENERATE_MASTER to redo it.")
elif generate_image(master_prompt(), refs, MASTER_PATH, match_master=False):
    print(f"✅ Master generated: {Image.open(MASTER_PATH).size[0]}x{Image.open(MASTER_PATH).size[1]}")
else:
    print("❌ Master generation failed - check the key / model and run again.")
if MASTER_PATH.exists():
    from IPython.display import display
    display(Image.open(MASTER_PATH).resize((600, int(600 * Image.open(MASTER_PATH).size[1] / Image.open(MASTER_PATH).size[0]))))


In [ ]:
#@title 5. Generate the house set (base, weather, Christmas)
if not MASTER_PATH.exists():
    raise SystemExit("❌ No master image. Run step 4 first.")

plan = []  # (filename, prompt)
for season in SEASONS:
    for t in TIMES:
        plan.append((f"{season}_{t}.png", variant_prompt(season, t)))
        for w in WEATHERS:
            if not GENERATE_RARE_WEATHER and (season, w) in UNLIKELY:
                continue
            plan.append((f"{season}_{w}_{t}.png", variant_prompt(season, t, w)))
if GENERATE_XMAS:
    for t in TIMES:
        plan.append((f"winter_xmas_{t}.png", variant_prompt("winter", t, "snowy", XMAS_EXTRAS[t])))

todo = [(f, p) for f, p in plan if not (OUT_DIR / f).exists()]
print(f"{len(plan)} house images in the plan, {len(plan) - len(todo)} already done, {len(todo)} to generate.")
failed = []
for i, (fname, prompt) in enumerate(todo, 1):
    print(f"[{i}/{len(todo)}] {fname}")
    if not generate_image(prompt, [MASTER_PATH], OUT_DIR / fname):
        failed.append(fname)

skipped = [f"{s}_{w}_{t}.png" for s in SEASONS for t in TIMES for w in WEATHERS if not GENERATE_RARE_WEATHER and (s, w) in UNLIKELY]
print(f"\n✅ Done. Failed: {failed or 'none'}")
if skipped:
    print(f"Skipped as rare weather (card falls back to the base image; tick GENERATE_RARE_WEATHER to make them): {', '.join(skipped)}")


In [ ]:
#@title 6. Generate vehicle overlays (optional)
#@markdown Each vehicle is added to a finished house image as a **local edit**, then the pixels that changed are
#@markdown cut out into a full-frame transparent PNG. Weather variants are edited onto a house image that already
#@markdown has that weather, and are confined to the clear-weather vehicle's silhouette (grown a little for snow on
#@markdown the roof), so the ground and bushes around the car never come along. Check step 8: a coverage well above
#@markdown ~10% means the model re-rendered more than the vehicle - delete that file and re-run, or set a
#@markdown `VEHICLE_N_REGION` so only changes in that part of the frame are kept.
REGENERATE_VEHICLES = False #@param {type:"boolean"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Tick to delete the existing vehicle overlays, raw renders and sprites and render them again.
REEXTRACT_ONLY = False #@param {type:"boolean"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Tick to keep the raw renders already made and only redo the cut-out (free - no API calls). Use this after changing extraction settings.
if not VEHICLES:
    print("No vehicle descriptions set - skipping.")
elif not MASTER_PATH.exists():
    raise SystemExit("❌ No master image. Run step 4 first.")
else:
    RAW_DIR = VEHICLE_DIR / "_raw"; RAW_DIR.mkdir(exist_ok=True)
    if REGENERATE_VEHICLES or REEXTRACT_ONLY:
        removed = 0
        for f in list(VEHICLE_DIR.glob("vehicle_*.png")) + list(SPRITE_DIR.glob("vehicle_*.png")) + (list(RAW_DIR.glob("*vehicle_*.png")) if REGENERATE_VEHICLES else []):
            f.unlink(); removed += 1
        print(f"Removed {removed} existing vehicle file(s) - {'regenerating' if REGENERATE_VEHICLES else 're-extracting from the kept raw renders'}.")
    weathers = ["sunny"] + (WEATHERS if VEHICLE_WEATHER_VARIANTS else [])
    vplan = []
    def base_for(w, t):
        """Weather variants are edited onto a house image that already has that weather, so snow or rain on
        the ground is in the base and only the vehicle registers as a change. Any season will do."""
        if w != "sunny":
            for s in ("summer", "autumn", "spring", "winter"):
                cand = OUT_DIR / f"{s}_{w}_{t}.png"
                if cand.exists(): return cand
        return OUT_DIR / f"summer_{t}.png"
    for n, (desc, spot, region) in enumerate(VEHICLES, 1):
        for t in TIMES:
            if not (OUT_DIR / f"summer_{t}.png").exists():
                print(f"⚠️ summer_{t}.png missing - run step 5 first; skipping vehicle {n} {t}"); continue
            for w in weathers:  # "sunny" comes first so its mask can confine the weather variants
                fname = f"vehicle_{n}_{t}.png" if w == "sunny" else f"vehicle_{n}_{w}_{t}.png"
                vplan.append((fname, desc, spot, region, t, w, base_for(w, t)))
    todo = [v for v in vplan if not (VEHICLE_DIR / v[0]).exists()]
    print(f"{len(vplan)} overlay images planned, {len(vplan) - len(todo)} done, {len(todo)} to generate.")
    for i, (fname, desc, spot, region, t, w, base) in enumerate(todo, 1):
        print(f"[{i}/{len(todo)}] {fname}")
        raw = RAW_DIR / fname
        if (REEXTRACT_ONLY and raw.exists()) or generate_image(vehicle_prompt(desc, spot, t, w), [base], raw):
            clear = VEHICLE_DIR / f"vehicle_{fname.split('_')[1]}_{t}.png"
            prior = shape_prior(clear, grow_pct=5.0) if (w != "sunny" and clear.exists()) else None
            cov, bbox, note = extract_overlay(raw, base, VEHICLE_DIR / fname, f"the {desc} (the vehicle) and its shadow",
                                              RAW_DIR / f"matte_{fname}", keep="largest", region=region, within=prior, reuse_matte=REEXTRACT_ONLY)
            if cov is None:
                print(f"   ⚠️ {note}"); continue
            flag = "  ⚠️ large area - more than the vehicle was kept" if cov > 10 else ("  ⚠️ almost nothing kept - no vehicle found" if cov < 0.2 else "")
            print(f"   {note}: {cov:.1f}% of the frame, region {bbox}{flag}")
    print("\n✅ Overlays done. Files are in output/overlays/.")
    print("\nSprites (cropped copies for positioned overlays) are in output/sprites/. Card config to place them where they were generated:")
    for n in range(1, len(VEHICLES) + 1):
        pos = make_sprites(f"vehicle_{n}_")
        if pos:
            print(f"  - entity: person.{n}\n    image: vehicle_{n}_{{weather}}_{{time}}.png\n    x: {pos['x']}\n    y: {pos['y']}\n    width: {pos['width']}")


In [ ]:
#@title 7. Generate gaming-mode images (optional, unused by the card)
if not GENERATE_GAMING:
    print("GENERATE_GAMING is off - skipping.")
else:
    for mode, desc in GAMING_MODES.items():
        fname = f"gaming_{mode}.png"
        if not (OUT_DIR / fname).exists():
            print(fname)
            generate_image(desc + "\n" + BASE_RULES, [MASTER_PATH], OUT_DIR / fname)
    print("✅ Gaming images done.")


In [ ]:
#@title 7b. Generate event images (optional)
#@markdown **overlay** mode writes transparent `event_{name}_{day|night}.png` into output/overlays/ (birthday: `event_birthday_{season}_{day|night}.png`),
#@markdown each cut out of a local edit of the matching season's house image. The comparison is done on blurred copies with a higher
#@markdown threshold, so the model's slight re-rendering of foliage and brick is ignored and only solid additions survive. If a result still
#@markdown carries scene fragments, raise `threshold` / `min_area_pct` in the `extract_changes` call in this cell. **full** mode writes complete house renders into output/.
REGENERATE_EVENTS = False #@param {type:"boolean"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Tick to delete the selected events' existing files and render them again (otherwise existing files are kept and skipped).
REEXTRACT_EVENTS_ONLY = False #@param {type:"boolean"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Overlay mode: keep the raw renders and only redo the cut-out (no API calls).
if not EVENTS_SELECTED:
    print("GENERATE_EVENTS is off or no events listed - skipping.")
elif not MASTER_PATH.exists():
    raise SystemExit("❌ No master image. Run step 4 first.")
else:
    unknown = [e for e in EVENTS_SELECTED if e not in EVENTS]
    if unknown:
        print(f"⚠️ Unknown events ignored: {unknown}. Available: {', '.join(EVENTS)}")
    eplan = []
    for name in EVENTS_SELECTED:
        if name not in EVENTS: continue
        season, extras = EVENTS[name]
        for s in (SEASONS if season is None else [season]):
            for t in TIMES:
                fname = f"event_{name}_{t}.png" if season else f"event_{name}_{s}_{t}.png"
                eplan.append((fname, s, t, extras))
    target_dir = VEHICLE_DIR if EVENT_MODE == "overlay" else OUT_DIR
    if REGENERATE_EVENTS or REEXTRACT_EVENTS_ONLY:
        for fname, *_ in eplan:
            targets = [target_dir / fname] + ([VEHICLE_DIR / "_raw" / fname, VEHICLE_DIR / "_raw" / f"matte_{fname}"] if REGENERATE_EVENTS else [])
            for f in targets:
                if f.exists(): f.unlink()
        print(f"Removed existing files for {', '.join(EVENTS_SELECTED)} - {'regenerating' if REGENERATE_EVENTS else 're-extracting from the kept raw renders'}.")
    # Full-scene renders from an earlier run share these filenames; park them so they are not mistaken for overlays
    if EVENT_MODE == "overlay":
        stale = [f for f in OUT_DIR.glob("event_*.png")]
        if stale:
            park = OUT_DIR / "_full_events"; park.mkdir(exist_ok=True)
            for f in stale: f.rename(park / f.name)
            print(f"ℹ️ Moved {len(stale)} full-scene event render(s) from output/ to output/_full_events/ (not zipped).")
    if EVENT_MODE == "overlay":
        RAW_DIR = VEHICLE_DIR / "_raw"; RAW_DIR.mkdir(exist_ok=True)
        todo = [e for e in eplan if not (VEHICLE_DIR / e[0]).exists()]
        done = [e[0] for e in eplan if (VEHICLE_DIR / e[0]).exists()]
        print(f"{len(eplan)} event overlays planned, {len(done)} already exist (kept - tick REGENERATE_EVENTS to redo), {len(todo)} to generate.")
        if done: print("   kept: " + ", ".join(done))
        for i, (fname, s, t, extras) in enumerate(todo, 1):
            base = OUT_DIR / f"{s}_{t}.png"
            if not base.exists():
                print(f"[{i}/{len(todo)}] {fname}: ⚠️ {base.name} missing - run step 5 first"); continue
            print(f"[{i}/{len(todo)}] {fname}")
            raw = RAW_DIR / fname
            if (REEXTRACT_EVENTS_ONLY and raw.exists()) or generate_image(event_overlay_prompt(t, extras), [base], raw):
                cov, bbox, note = extract_overlay(raw, base, VEHICLE_DIR / fname, f"the decorations that were added to the scene ({extras[:160]}...)",
                                                  RAW_DIR / f"matte_{fname}", keep="all", reuse_matte=REEXTRACT_EVENTS_ONLY)
                if cov is None:
                    print(f"   ⚠️ {note}"); continue
                flag = "  ⚠️ most of the frame kept - the scene was not erased" if cov > 40 else ("  ⚠️ almost nothing kept" if cov < 0.2 else "")
                print(f"   {note}: {cov:.1f}% of the frame, spread {bbox}{flag}")
        print("\n✅ Event overlays done. Files are in output/overlays/.")
    else:
        todo = [e for e in eplan if not (OUT_DIR / e[0]).exists()]
        print(f"{len(eplan)} full event images planned, {len(eplan) - len(todo)} already exist (kept), {len(todo)} to generate.")
        for i, (fname, s, t, extras) in enumerate(todo, 1):
            print(f"[{i}/{len(todo)}] {fname}")
            generate_image(event_prompt(s, t, extras), [MASTER_PATH], OUT_DIR / fname)
        print("\n✅ Event images done.")


In [ ]:
#@title 8. Review thumbnails
#@markdown Overlays are shown composited over the matching house image so you can check alignment.
from IPython.display import display

def sheet(paths, cols=4, w=320, bg=None):
    paths = [p for p in paths if p.exists()]
    if not paths: return None
    first = Image.open(paths[0]); h = int(w * first.size[1] / first.size[0])
    rows = (len(paths) + cols - 1) // cols
    canvas = Image.new("RGB", (cols * w, rows * (h + 18)), (18, 18, 18))
    try:
        from PIL import ImageDraw
        draw = ImageDraw.Draw(canvas)
    except Exception:  # noqa: BLE001 - a half-upgraded Pillow can break ImageDraw; labels are optional
        draw = None
    for i, p in enumerate(paths):
        img = Image.open(p).convert("RGBA")
        if bg is not None:
            base = Image.open(bg(p.name)).convert("RGBA") if Path(bg(p.name)).exists() else Image.new("RGBA", img.size, (40, 40, 40, 255))
            img = Image.alpha_composite(base, img)
        img = img.convert("RGB").resize((w, h))
        x, y = (i % cols) * w, (i // cols) * (h + 18)
        canvas.paste(img, (x, y + 18))
        if draw: draw.text((x + 4, y + 2), p.name, fill=(230, 230, 230))
        else: print(f"  [{i}] {p.name}")
    return canvas

houses = sorted(OUT_DIR.glob("*.png"))
print(f"{len(houses)} house images"); s = sheet(houses); display(s) if s else None

def house_for(overlay_name):
    m = re.match(r"vehicle_\d+(?:_(\w+?))?_(day|night)\.png", overlay_name)
    if m:
        w, t = m.group(1), m.group(2)
        cand = OUT_DIR / (f"summer_{w}_{t}.png" if w else f"summer_{t}.png")
        return cand if cand.exists() else OUT_DIR / f"summer_{t}.png"
    m = re.match(r"event_(\w+?)(?:_(winter|spring|summer|autumn))?_(day|night)\.png", overlay_name)
    if m:
        name, season, t = m.groups()
        season = season or (EVENTS.get(name, ("summer", ""))[0] or "summer")
        return OUT_DIR / f"{season}_{t}.png"
    return OUT_DIR / "summer_day.png"

overlays = sorted(VEHICLE_DIR.glob("vehicle_*.png")) + sorted(VEHICLE_DIR.glob("event_*.png"))
print(f"{len(overlays)} overlay images"); s = sheet(overlays, bg=house_for); display(s) if s else None
sprites = sorted(SPRITE_DIR.glob("*.png"))
if sprites:
    print(f"{len(sprites)} sprites"); s = sheet(sprites, cols=6, w=200); display(s) if s else None
import numpy as np
for p in overlays:
    a = np.asarray(Image.open(p).convert("RGBA"))[..., 3] > 127
    ys, xs = np.where(a); H, W = a.shape
    if len(xs):
        print(f"  {p.name}: {a.mean()*100:.1f}% opaque, region {round(xs.min()/W*100)},{round(ys.min()/H*100)},{round(xs.max()/W*100)},{round(ys.max()/H*100)}")

# Both vehicles at once (the "everyone home" case) - check they sit in separate spots
for t in TIMES:
    parts = [VEHICLE_DIR / f"vehicle_{n}_{t}.png" for n in range(1, len(VEHICLES) + 1)]
    base = OUT_DIR / f"summer_{t}.png"
    if len(parts) > 1 and all(p.exists() for p in parts) and base.exists():
        img = Image.open(base).convert("RGBA")
        for p in parts:
            img = Image.alpha_composite(img, Image.open(p).convert("RGBA"))
        print(f"All vehicles, {t}:"); display(img.convert("RGB").resize((640, int(640 * img.size[1] / img.size[0]))))


In [ ]:
#@title 9. Download everything as a zip
#@markdown Copy the contents of `house_card_images.zip` into the card's image folder (default `config/www/community/house_card/images/`).
zip_path = BASE_DIR / "house_card_images.zip"
overlay_files = list(VEHICLE_DIR.glob("vehicle_*.png")) + list(VEHICLE_DIR.glob("event_*.png"))
overlay_names = {f.name for f in overlay_files}
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUT_DIR.glob("*.png"):
        if f.name in overlay_names:
            print(f"  skipping output/{f.name}: an overlay with the same name takes precedence"); continue
        zf.write(f, f.name)
    for f in overlay_files: zf.write(f, f.name)
    for f in SPRITE_DIR.glob("*.png"): zf.write(f, f"sprites/{f.name}")
    if MASTER_PATH.exists(): zf.write(MASTER_PATH, "_master_reference.png")
n = len(list(OUT_DIR.glob("*.png"))) + len(list(VEHICLE_DIR.glob("vehicle_*.png"))) + len(list(VEHICLE_DIR.glob("event_*.png")))
print(f"Zipped {n} images.")
files.download(str(zip_path))
